In [ ]:
"""
ml_matchmaking.py
==================
Two simple ML algorithms working together for team matchmaking:

ALGORITHM 1: K-MEANS CLUSTERING
    - Groups all 100 players into 6 skill clusters using their stats
      (Level, K/D, Feedback, Win Rate, Achievements, ELO).
    - This replaces manually-assigned "Tier" with a data-driven tier.

ALGORITHM 2: LOGISTIC REGRESSION
    - Predicts win probability of Team A vs Team B, using team stats.
    - We try every possible 4v4 split of 8 players and pick the split
      where win probability is closest to 50/50 (fairest match).

Since we don't have real match history, we SIMULATE past matches to
train the logistic regression model. This is clearly labeled as
simulated data, not real results.
"""

import itertools
import subprocess
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")   # hide sklearn's internal technical warnings
                                     # (we handle the important cases ourselves,
                                     # e.g. the accuracy check below)

# NOTE: no np.random.seed() here on purpose -- we WANT different
# random matches / different players every time you run this script.

EXCEL_PATH = "Players.xlsx"
RECALC_SCRIPT = "/mnt/skills/public/xlsx/scripts/recalc.py"


def refresh_online_status():
    """
    Excel's RAND() formula only updates when the file is actually
    recalculated (e.g. opening it in Excel and pressing F9).
    pandas.read_excel() just reads whatever values were saved LAST time,
    so Online Status looked frozen. This forces a real recalculation
    (via LibreOffice) before we read the file, so RAND() rerolls.
    """
    subprocess.run(["python3", RECALC_SCRIPT, EXCEL_PATH],
                    capture_output=True, text=True)


# =========================================================================
# STEP 1: LOAD DATA
# =========================================================================
def load_online_players():
    refresh_online_status()   # reroll RAND() so Online/Offline actually changes
    df = pd.read_excel(EXCEL_PATH, sheet_name="Players")
    online_df = df[df["Online Status"] == "Online"].reset_index(drop=True)
    print(f"Loaded {len(online_df)} online players.\n")
    return online_df


# =========================================================================
# ALGORITHM 1: K-MEANS CLUSTERING
# Groups players into skill clusters based on their stats.
# =========================================================================
def cluster_players(df, k=6):
    # Features used to decide skill clusters
    features = ["Level", "K/D Ratio", "Peer Feedback (1-5)",
                "Win Rate", "Event Achievements", "ELO Rating"]

    X = df[features].values

    # Scale features so no single column (like ELO, which has big numbers)
    # dominates the distance calculation.
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Run K-Means with k=6 clusters (same as our 6 tiers: Bronze..Master)
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    df["Cluster"] = kmeans.fit_predict(X_scaled)

    # Rename clusters 0,1,2... into skill order (low ELO -> high ELO)
    cluster_avg_elo = df.groupby("Cluster")["ELO Rating"].mean().sort_values()
    rank_map = {old: new for new, old in enumerate(cluster_avg_elo.index)}
    df["SkillCluster"] = df["Cluster"].map(rank_map)

    print("STEP 1: K-MEANS CLUSTERING RESULT")
    print("-" * 40)
    summary = df.groupby("SkillCluster")["ELO Rating"].agg(["count", "mean"])
    print(summary.round(1))
    print()
    return df


# =========================================================================
# STEP 2: SIMULATE PAST MATCHES (needed to train logistic regression,
# since we have no real win/loss history)
# =========================================================================
def simulate_matches(df, n_matches=2000):
    rows = []
    players = df.to_dict("records")

    for _ in range(n_matches):
        # Pick 8 random players, split into two random teams of 4
        eight = list(np.random.choice(len(players), 8, replace=False))
        team_a_idx = eight[:4]
        team_b_idx = eight[4:]

        team_a = [players[i] for i in team_a_idx]
        team_b = [players[i] for i in team_b_idx]

        elo_a = np.mean([p["ELO Rating"] for p in team_a])
        elo_b = np.mean([p["ELO Rating"] for p in team_b])
        kd_a = np.mean([p["K/D Ratio"] for p in team_a])
        kd_b = np.mean([p["K/D Ratio"] for p in team_b])
        fb_a = np.mean([p["Peer Feedback (1-5)"] for p in team_a])
        fb_b = np.mean([p["Peer Feedback (1-5)"] for p in team_b])

        # Simple simulation rule: higher ELO team is MORE LIKELY to win,
        # but we add randomness so it's not 100% guaranteed (like real games)
        elo_diff = elo_a - elo_b
        win_chance_a = 1 / (1 + np.exp(-elo_diff / 200))  # sigmoid curve
        team_a_wins = 1 if np.random.rand() < win_chance_a else 0

        rows.append({
            "elo_diff": elo_a - elo_b,
            "kd_diff": kd_a - kd_b,
            "feedback_diff": fb_a - fb_b,
            "team_a_wins": team_a_wins
        })

    match_df = pd.DataFrame(rows)
    print("STEP 2: SIMULATED MATCH DATA (for training only)")
    print("-" * 40)
    print(f"Generated {n_matches} simulated matches.")
    print(match_df.head(3))
    print()
    return match_df


# =========================================================================
# ALGORITHM 2: LOGISTIC REGRESSION
# Predicts win probability of Team A based on team stat differences.
# =========================================================================
def train_win_predictor(match_df):
    X = match_df[["elo_diff", "kd_diff", "feedback_diff"]]
    y = match_df["team_a_wins"]

    # Simple train/test split to check accuracy
    split = int(len(X) * 0.8)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    model = LogisticRegression()
    model.fit(X_train, y_train)

    accuracy = accuracy_score(y_test, model.predict(X_test))

    print("STEP 3: LOGISTIC REGRESSION TRAINING RESULT")
    print("-" * 40)
    print(f"Model accuracy on test data: {accuracy:.2%}")

    # -------------------------------------------------------------------
    # CHECK: is the model actually good, or basically guessing randomly?
    # A model that is right ~50% of the time on a YES/NO question is
    # no better than flipping a coin -- usually happens when players'
    # stats are too similar (little difference for the model to learn from).
    # -------------------------------------------------------------------
    if accuracy < 0.60:
        print("WARNING: model accuracy is LOW (below 60%).")
        print("Predictions below this point may be unreliable -- this")
        print("usually happens when the online player pool has very")
        print("similar stats, giving the model little signal to learn from.")
    print()
    return model


# =========================================================================
# STEP 4: FIND THE FAIREST 4v4 SPLIT
# Try every possible way to split 8 players into 2 teams of 4.
# Pick the split where win probability is closest to 50/50.
# =========================================================================
def find_fairest_split(eight_players, model):
    best_split = None
    best_diff_from_half = 1.0   # start with worst possible value

    player_indices = list(range(8))

    # itertools.combinations gives every unique way to pick 4 out of 8
    for combo in itertools.combinations(player_indices, 4):
        team_a_idx = list(combo)
        team_b_idx = [i for i in player_indices if i not in team_a_idx]

        team_a = [eight_players[i] for i in team_a_idx]
        team_b = [eight_players[i] for i in team_b_idx]

        elo_diff = np.mean([p["ELO Rating"] for p in team_a]) - \
                   np.mean([p["ELO Rating"] for p in team_b])
        kd_diff = np.mean([p["K/D Ratio"] for p in team_a]) - \
                  np.mean([p["K/D Ratio"] for p in team_b])
        fb_diff = np.mean([p["Peer Feedback (1-5)"] for p in team_a]) - \
                  np.mean([p["Peer Feedback (1-5)"] for p in team_b])

        X_new = pd.DataFrame([[elo_diff, kd_diff, fb_diff]],
                              columns=["elo_diff", "kd_diff", "feedback_diff"])
        win_prob_a = model.predict_proba(X_new)[0][1]

        diff_from_half = abs(win_prob_a - 0.5)

        if diff_from_half < best_diff_from_half:
            best_diff_from_half = diff_from_half
            best_split = (team_a, team_b, win_prob_a)

    return best_split


# =========================================================================
# VISUALIZATION: overlapping line graph -- Team A vs Team B
# Combines each player's stats into ONE overall strength score, so the
# whole team comparison fits in a single, easy-to-read line chart.
# =========================================================================
def plot_team_comparison(team_a, team_b, win_prob_a, outfile="fairest_match_teams.png"):
    slots = [1, 2, 3, 4]   # Player 1..4 in each team

    def overall_score(p):
        # Combine stats into one score. ELO dominates (main skill signal),
        # the rest add smaller adjustments -- similar to the ELO formula
        # used earlier in the project.
        return (p["ELO Rating"]
                + p["K/D Ratio"] * 100
                + p["Peer Feedback (1-5)"] * 50
                + p["Win Rate"] * 200)

    scores_a = [overall_score(p) for p in team_a]
    scores_b = [overall_score(p) for p in team_b]

    plt.figure(figsize=(8, 5))
    plt.plot(slots, scores_a, marker="o", linewidth=2.5, color="#1f77b4", label="Team A")
    plt.plot(slots, scores_b, marker="s", linewidth=2.5, color="#d62728", label="Team B")
    plt.fill_between(slots, scores_a, scores_b, color="grey", alpha=0.1)

    plt.xticks(slots, [f"Player {i}" for i in slots])
    plt.ylabel("Overall Strength Score")
    plt.title(f"Team A vs Team B -- Overall Strength\n"
              f"Predicted win probability: Team A {win_prob_a:.1%} | Team B {1-win_prob_a:.1%}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(outfile, dpi=150)
    plt.close()
    print(f"Saved chart -> {outfile}")


# =========================================================================
# MAIN
# =========================================================================
def main():
    df = load_online_players()

    # -------------------------------------------------------------------
    # CHECK #1: do we even have enough online players to run this at all?
    # (must happen BEFORE clustering -- K-Means needs at least k=6
    # players to form 6 clusters, otherwise it crashes)
    # -------------------------------------------------------------------
    online_count = len(df)

    if online_count < 8:
        print("SORRY -- only", online_count, "players online.")
        print("Minimum 8 players needed for a 4v4 match.")
        print("Please try again later, or switch to another game mode.")
        return   # stop here, nothing else can run

    if online_count < 12:
        print(f"WARNING: only {online_count} online players (below the ideal 12).")
        print("Match quality may be POOR -- fewer options to balance skill.\n")

    df = cluster_players(df)

    match_history = simulate_matches(df)
    model = train_win_predictor(match_history)

    # Pick 8 RANDOM online players as this run's match pool
    # (was df.head(8) before -- always picked the same 8 people)
    sample_df = df.sample(n=8).reset_index(drop=True)
    eight_players = sample_df.to_dict("records")

    team_a, team_b, win_prob_a = find_fairest_split(eight_players, model)

    print("STEP 4: FAIREST 4v4 MATCH FOUND")
    print("-" * 40)
    print("Team A:")
    for p in team_a:
        print(f"   - {p['Name']} | ELO {p['ELO Rating']:.0f} | Skill Cluster {p['SkillCluster']}")
    print("Team B:")
    for p in team_b:
        print(f"   - {p['Name']} | ELO {p['ELO Rating']:.0f} | Skill Cluster {p['SkillCluster']}")
    print()
    print(f"Predicted win probability -> Team A: {win_prob_a:.1%} | Team B: {1-win_prob_a:.1%}")

    plot_team_comparison(team_a, team_b, win_prob_a)


if __name__ == "__main__":
    main()

Loaded 66 online players.

STEP 1: K-MEANS CLUSTERING RESULT
----------------------------------------
              count    mean
SkillCluster               
0                12  1133.0
1                14  1208.4
2                14  1589.0
3                14  2020.9
4                 6  2417.8
5                 6  2438.2

STEP 2: SIMULATED MATCH DATA (for training only)
----------------------------------------
Generated 2000 simulated matches.
   elo_diff  kd_diff  feedback_diff  team_a_wins
0  -103.300   0.3325         -0.925            0
1   118.575   0.1200         -0.250            1
2   931.350   0.8850          0.250            1

STEP 3: LOGISTIC REGRESSION TRAINING RESULT
----------------------------------------
Model accuracy on test data: 77.75%

STEP 4: FAIREST 4v4 MATCH FOUND
----------------------------------------
Team A:
   - Dev Bose | ELO 1179 | Skill Cluster 1
   - Vihaan Verma | ELO 1199 | Skill Cluster 0
   - Rohan Patel | ELO 1500 | Skill Cluster 2
   - Manish V